In [188]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [189]:
df = pd.read_csv('test.txt',sep=";",header=None,names=['text','emotion'])
df.head()

,text,emotion
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness


In [190]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [191]:
emotions = df['emotion'].unique()

In [192]:
emo_map = {}
for i, emotion in enumerate(emotions):
    emo_map[emotion] = i
emo_map

{'sadness': 0, 'joy': 1, 'fear': 2, 'anger': 3, 'love': 4, 'surprise': 5}

In [193]:
df['emotion'] = df['emotion'].map(emo_map)
df.head()

,text,emotion
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


### Text Lowercasing

In [194]:
df['text'] = df['text'].apply(lambda x: x.lower())

### Text Cleaning

* Removing Punctuations
* Removing Numbers
* Removing URLs/Links
* Removing HTML Tags (if web scraping is done)
* Removing Emojis & Special chars
* Removing Stopwords

In [195]:
#creating function
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

In [196]:
df['text'] = df['text'].apply(remove_punc)

In [197]:
def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new+i
    return new

In [198]:
df['text'] = df['text'].apply(remove_num)

In [199]:
def remove_emo(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new = new+i
    return new

In [200]:
df['text'] = df['text'].apply(remove_emo)

### Removing Stopwords

via NLTK

In [201]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [202]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [203]:
stop_words = set(stopwords.words('english'))

In [204]:
def remove_stop(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if i not in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

In [205]:
df['text'] = df['text'].apply(remove_stop)

In [206]:
df.loc[0]['text']

'im feeling rather rotten im ambitious right'

In [207]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'],df['emotion'],test_size=0.2, random_state=42)

In [208]:
vectorizer_bow = CountVectorizer()

X_train_bow = vectorizer_bow.fit_transform(X_train)
X_test_bow = vectorizer_bow.transform(X_test)

In [209]:
vectorizer_tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=20000,
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

In [210]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

y_pred_bow = nb_bow.predict(X_test_bow)

print("Accuracy of BOW via NBayes: ", accuracy_score(y_test, y_pred_bow)*100)

Accuracy of BOW via NBayes:  61.5


In [211]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

print("Accuracy of Tf-Idf via NBayes: ", accuracy_score(y_test, y_pred_tfidf)*100)

Accuracy of Tf-Idf via NBayes:  59.25


In [212]:
from sklearn.linear_model import LogisticRegression

lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)

y_pred_bow_lr = lr_bow.predict(X_test_bow)

print("Accuracy of BOW via LRegression: ", accuracy_score(y_test, y_pred_bow_lr)*100)

Accuracy of BOW via LRegression:  66.5


In [213]:
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf_lr = lr_tfidf.predict(X_test_tfidf)

print("Accuracy of Tf-Idf via LRegression: ", accuracy_score(y_test, y_pred_tfidf_lr)*100)

Accuracy of Tf-Idf via LRegression:  64.25


In [214]:
from sklearn.svm import SVC

svm_bow = SVC(kernel='linear')
svm_bow.fit(X_train_bow, y_train)
svm_bow.fit(X_train_bow, y_train)

y_pred_bow_svc = svm_bow.predict(X_test_bow)

print("Accuracy of BOW via SVC: ", accuracy_score(y_test, y_pred_bow_svc)*100)

Accuracy of BOW via SVC:  71.0


In [215]:
svm_tfidf = SVC(kernel='linear',C=1.5)
svm_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf_svc = svm_tfidf.predict(X_test_tfidf)

print("Accuracy of Tf-Idf via SVC: ", accuracy_score(y_test, y_pred_tfidf_svc)*100)

Accuracy of Tf-Idf via SVC:  74.5
